# 09 — Process manually identified Commission decisions

## Purpose

This notebook converts the raw source documents acquired by
`08_download_com_manual.ipynb` into **one canonical UTF-8 text file per source file**.

It is designed as a simple, reproducible processing step for scientific data publication:

1. use notebook 08's download manifest as the authoritative handoff;
2. process only successfully acquired local source files;
3. verify that the raw file still matches its recorded SHA-256;
4. extract HTML and plain-text files directly;
5. process PDFs with the deterministic fallback  
   **PyMuPDF → pdftotext → MinerU OCR**;
6. use a small set of transparent technical checks only to decide whether a fallback is needed;
7. save one canonical `.txt` file in `data/processed/com_manual/text/`;
8. write one compact clean-file manifest linking the raw and processed files.

No Markdown document copies, candidate directories, candidate-score tables, or
regex/LLM-specific text variants are produced.

The aim is to preserve a clear distinction between **raw acquisition** and
**text processing** while keeping the audit trail understandable.

## Inputs and outputs

### Input

Authoritative handoff from notebook 08:

`output/com_manual/com_manual_download_manifest.csv`

Only rows with `success == True` and a local `raw_file_path` enter text processing.

### Outputs

Canonical text files:

`data/processed/com_manual/text/<raw_basename>.txt`

For example:

`31982D0465.html → 31982D0465.txt`

Publication-oriented clean-file manifest:

`output/com_manual/com_manual_clean_file_manifest.csv`

Operational checkpoint:

`logs/com_manual/processing_state.jsonl`

The JSONL checkpoint exists only so interrupted OCR runs can resume safely. It is not
intended as a separate analytical or publication dataset.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from bs4 import BeautifulSoup, UnicodeDammit
except Exception:
    BeautifulSoup = None
    UnicodeDammit = None

try:
    from charset_normalizer import from_bytes as charset_from_bytes
except Exception:
    charset_from_bytes = None


## 1. Repository paths and processing settings

The notebook locates the repository from the notebook's working directory. If automatic
detection fails, set the `ECCJEU_PROJECT_ROOT` environment variable or uncomment the
manual override.

The thresholds below are deliberately few and interpretable. They are **technical
fallback checks**, not a substantive measure of legal-document quality.

In [2]:
# ---------------------------------------------------------------------
# Repository paths
# ---------------------------------------------------------------------

def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(
                f"ECCJEU_PROJECT_ROOT does not exist: {root}"
            )
        return root

    start = (start or Path.cwd()).resolve()

    # Best marker for this notebook: output produced by notebook 08.
    for candidate in [start, *start.parents]:
        if (
            candidate
            / "output"
            / "com_manual"
            / "com_manual_download_manifest.csv"
        ).exists():
            return candidate

    # Repository-layout fallbacks.
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "01_commission").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Run the notebook from within "
        "the repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

INPUT_MANIFEST = (
    PROJECT_ROOT
    / "output"
    / "com_manual"
    / "com_manual_download_manifest.csv"
)

PROCESSED_TEXT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "com_manual"
    / "text"
)

OUTPUT_DIR = PROJECT_ROOT / "output" / "com_manual"
LOG_DIR = PROJECT_ROOT / "logs" / "com_manual"

CLEAN_FILE_MANIFEST = (
    OUTPUT_DIR
    / "com_manual_clean_file_manifest.csv"
)

PROCESSING_STATE_JSONL = (
    LOG_DIR
    / "processing_state.jsonl"
)

for directory in [
    PROCESSED_TEXT_DIR,
    OUTPUT_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Processing configuration
# ---------------------------------------------------------------------

PROCESSING_VERSION = "1.0"
FORCE_REPROCESS = False

RUN_PDFTOTEXT = True
RUN_OCR = True

PDFTOTEXT_TIMEOUT_SECONDS = 300
OCR_TIMEOUT_SECONDS = 900

# Transparent technical checks used only to decide whether another
# extraction method should be attempted.
MIN_TOTAL_CHARS = 200
MIN_PDF_CHARS_PER_PAGE = 500
MAX_BAD_ENCODING_MARKERS = 3
MAX_ONE_CHAR_TOKEN_RATIO = 0.20

BAD_ENCODING_MARKERS = [
    "�",
    "Ã",
    "Â",
    "â€™",
    "â€œ",
    "â€",
    "ðŸ",
]

SUPPORTED_FILE_FORMATS = {
    "pdf",
    "html",
    "htm",
    "xhtml",
    "xml",
    "txt",
    "text",
}

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")


def repo_path(path: Path) -> str:
    """Portable path relative to the repository root."""
    return (
        path.resolve()
        .relative_to(PROJECT_ROOT.resolve())
        .as_posix()
    )


def resolve_repo_path(value: str) -> Path:
    """Resolve repository-relative paths written by notebook 08."""
    value = str(value or "").strip().replace("\\", "/")

    if not value:
        raise ValueError("Empty repository path.")

    path = Path(value)

    if path.is_absolute():
        # Accept an absolute path only when it points inside this repository.
        resolved = path.resolve()
        try:
            resolved.relative_to(PROJECT_ROOT.resolve())
        except ValueError as exc:
            raise ValueError(
                f"Path lies outside the repository: {value}"
            ) from exc
        return resolved

    if ".." in path.parts:
        raise ValueError(
            f"Non-portable repository path: {value}"
        )

    # Also tolerate older manifests prefixed with the repository directory name.
    if path.parts and path.parts[0] in {
        PROJECT_ROOT.name,
        "ec_cjeu",
        "eccjeu",
    }:
        path = Path(*path.parts[1:])

    return PROJECT_ROOT / path


def utc_now_iso() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
    )


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


print("Project root: .")
print(
    "Input manifest:",
    "output/com_manual/com_manual_download_manifest.csv",
)
print(
    "Processed text:",
    "data/processed/com_manual/text/",
)
print(
    "Clean-file manifest:",
    "output/com_manual/com_manual_clean_file_manifest.csv",
)
print(
    "Processing state:",
    "logs/com_manual/processing_state.jsonl",
)
print("PyMuPDF available:", fitz is not None)
print("pdftotext available:", PDFTOTEXT_CLI is not None)
print("MinerU available:", MINERU_CLI is not None)
print("BeautifulSoup available:", BeautifulSoup is not None)
print("charset-normalizer available:", charset_from_bytes is not None)


Project root: .
Input manifest: output/com_manual/com_manual_download_manifest.csv
Processed text: data/processed/com_manual/text/
Clean-file manifest: output/com_manual/com_manual_clean_file_manifest.csv
Processing state: logs/com_manual/processing_state.jsonl
PyMuPDF available: True
pdftotext available: True
MinerU available: True
BeautifulSoup available: True
charset-normalizer available: True


### Optional dependencies

The notebook uses:

- **PyMuPDF** for native PDF text extraction;
- **Poppler `pdftotext`** as a second native PDF extractor;
- **MinerU** as the OCR fallback;
- **BeautifulSoup** and **charset-normalizer** for HTML/text decoding.

Install missing dependencies in the environment rather than writing environment-specific
details into the publication manifest.

In [3]:
# Uncomment only if required in your environment:
#
# %pip install -U pymupdf beautifulsoup4 charset-normalizer
#
# Ubuntu/Debian:
# !sudo apt-get update && sudo apt-get install -y poppler-utils
#
# MinerU installation depends on the environment used for OCR.


## 2. Validate the handoff from notebook 08

Notebook 09 does not reconstruct URLs or redownload documents. Acquisition belongs to
notebook 08.

The checks here are intentionally simple:

- the required manifest columns exist;
- only `success == True` rows are processed;
- each referenced raw file exists;
- the recorded raw SHA-256 is well formed;
- the current raw file still matches that SHA-256;
- replacing the raw extension with `.txt` will not create duplicate processed filenames.

Rows that failed during acquisition remain the responsibility of notebook 08 and are not
duplicated as text-processing failures.

In [4]:
REQUIRED_COLUMNS = {
    "source_row",
    "url_index_in_row",
    "decision_date",
    "case_numbers_raw",
    "case_numbers_normalized",
    "decision_label",
    "decision_title",
    "celex",
    "source_url",
    "file_name",
    "raw_file_path",
    "success",
    "file_format",
    "sha256",
}


def read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input manifest is missing: {path}"
        )

    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
    )


def as_bool(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


download_manifest = read_manifest(INPUT_MANIFEST)

missing_columns = sorted(
    REQUIRED_COLUMNS - set(download_manifest.columns)
)

if missing_columns:
    raise ValueError(
        "The notebook 08 manifest is missing required columns: "
        f"{missing_columns}"
    )


successful = download_manifest.loc[
    as_bool(download_manifest["success"])
    & download_manifest["raw_file_path"].ne("")
].copy()

if successful.empty:
    raise ValueError(
        "No successfully downloaded COM manual files are available "
        "for text processing."
    )


raw_paths = [
    resolve_repo_path(value)
    for value in successful["raw_file_path"]
]

missing_files = [
    successful.iloc[index]["file_name"]
    for index, path in enumerate(raw_paths)
    if not path.exists() or not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} referenced raw file(s) are missing; "
        f"examples: {missing_files[:10]}"
    )


bad_hash = ~successful["sha256"].str.fullmatch(
    r"[0-9a-fA-F]{64}"
)

if bad_hash.any():
    examples = successful.loc[
        bad_hash,
        ["file_name", "sha256"],
    ].head(10)

    raise ValueError(
        "Successful download rows contain malformed SHA-256 values: "
        f"{examples.to_dict('records')}"
    )


def normalize_file_format(
    manifest_value: str,
    raw_path: Path,
) -> str:
    value = str(manifest_value or "").strip().lower().lstrip(".")

    if value in {"htm", "html"}:
        return value

    if value in {"text", "txt"}:
        return value

    if value in SUPPORTED_FILE_FORMATS:
        return value

    suffix = raw_path.suffix.lower().lstrip(".")

    if suffix in SUPPORTED_FILE_FORMATS:
        return suffix

    return value or suffix or "unknown"


successful["raw_path_internal"] = [
    str(path)
    for path in raw_paths
]

successful["raw_file_name"] = [
    path.name
    for path in raw_paths
]

successful["file_format_normalized"] = [
    normalize_file_format(fmt, path)
    for fmt, path in zip(
        successful["file_format"],
        raw_paths,
    )
]

successful["raw_sha256"] = (
    successful["sha256"]
    .str.lower()
)


def expected_text_name(raw_path: Path) -> str:
    return raw_path.with_suffix(".txt").name


successful["clean_file_name"] = [
    expected_text_name(path)
    for path in raw_paths
]

if not successful["clean_file_name"].is_unique:
    duplicates = (
        successful["clean_file_name"]
        .value_counts()
        .loc[lambda s: s > 1]
        .index
        .tolist()
    )

    raise ValueError(
        "Changing raw-file extensions to .txt would create duplicate "
        f"processed filenames: {duplicates[:10]}"
    )


successful["record_id"] = [
    f"com_manual__row{int(row):05d}_url{int(url_index):02d}"
    for row, url_index in zip(
        successful["source_row"],
        successful["url_index_in_row"],
    )
]

if not successful["record_id"].is_unique:
    raise ValueError(
        "The successful download manifest contains duplicate source-row/URL records."
    )


work_manifest = (
    successful
    .sort_values(
        ["source_row", "url_index_in_row"]
    )
    .reset_index(drop=True)
)


print(f"Rows in notebook 08 manifest: {len(download_manifest):,}")
print(f"Files available for processing: {len(work_manifest):,}")
print(
    "Rows not processed here:",
    len(download_manifest) - len(work_manifest),
)

display(
    work_manifest[
        [
            "record_id",
            "decision_title",
            "celex",
            "raw_file_name",
            "file_format_normalized",
            "raw_file_path",
        ]
    ].head(20)
)


Rows in notebook 08 manifest: 16
Files available for processing: 16
Rows not processed here: 0


,record_id,decision_title,celex,raw_file_name,file_format_normalized,raw_file_path
0,com_manual__row00001_url01,National Panasonic France,31982D0465,31982D0465.html,html,data/raw/com_manual/files/31982D0465.html
1,com_manual__row00010_url01,GSM Spain,31997D0181,31997D0181.html,html,data/raw/com_manual/files/31997D0181.html
2,com_manual__row00011_url01,Additional implementation periods to Portugal,31997D0310,31997D0310.html,html,data/raw/com_manual/files/31997D0310.html
3,com_manual__row00012_url01,Additional implementation periods to Luxembourg,31997D0568,31997D0568.html,html,data/raw/com_manual/files/31997D0568.html
4,com_manual__row00013_url01,Additional implementation periods to Spain,31997D0603,31997D0603.html,html,data/raw/com_manual/files/31997D0603.html
5,com_manual__row00014_url01,Broadcasting television Flanders,31997D0606,31997D0606.pdf,pdf,data/raw/com_manual/files/31997D0606.pdf
6,com_manual__row00015_url01,Additional implementation periods to Greece,31997D0607,31997D0607.html,html,data/raw/com_manual/files/31997D0607.html
7,com_manual__row00016_url01,Italian ports legislation | Tariffs for piloti...,31997D0744,31997D0744.html,html,data/raw/com_manual/files/31997D0744.html
8,com_manual__row00002_url01,Anheuser-Busch Inc/Scottish and New Castle,32000D0146,32000D0146.html,html,data/raw/com_manual/files/32000D0146.html
9,com_manual__row00003_url01,Zaventem,31995D0364,31995D0364.html,html,data/raw/com_manual/files/31995D0364.html


## 3. Canonical text representation

The processed text is intended to be a **general reusable representation** of the source
document, not a version optimized for a particular parser or language model.

Cleaning is therefore deliberately light:

- Unicode is normalized with NFKC;
- byte-order marks, zero-width spaces and soft hyphens are removed;
- line endings and ordinary whitespace are normalized;
- words split only by line-ending hyphenation are joined;
- paragraph structure is otherwise preserved;
- a few obvious Markdown markers are removed when MinerU returns Markdown internally.

Only one UTF-8 `.txt` representation is retained.

In [5]:
def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize(
        "NFKC",
        text or "",
    )

    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
        "\f": "\n\n",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


def markdown_to_plain_text(text: str) -> str:
    """
    Conservative removal of common Markdown syntax from MinerU output.
    Legal wording is retained; only presentation markup is stripped.
    """
    text = text or ""

    # Images: keep alt text, discard target.
    text = re.sub(
        r"!\[([^\]]*)\]\([^)]+\)",
        r"\1",
        text,
    )

    # Links: keep visible text.
    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text,
    )

    # Headings and block quotes.
    text = re.sub(
        r"(?m)^\s{0,3}#{1,6}\s+",
        "",
        text,
    )
    text = re.sub(
        r"(?m)^\s{0,3}>\s?",
        "",
        text,
    )

    # Fenced-code markers and horizontal rules.
    text = re.sub(
        r"(?m)^\s*```[A-Za-z0-9_-]*\s*$",
        "",
        text,
    )
    text = re.sub(
        r"(?m)^\s*(?:---+|\*\*\*+|___+)\s*$",
        "",
        text,
    )

    # Light inline emphasis cleanup.
    text = text.replace("**", "")
    text = text.replace("__", "")
    text = text.replace("`", "")

    return text


def clean_text(text: str) -> str:
    """Light normalization while preserving readable document structure."""
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")

    # Join words broken by line-ending hyphenation.
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip() + "\n" if text.strip() else ""


def count_bad_encoding_markers(text: str) -> int:
    return sum(
        text.count(marker)
        for marker in BAD_ENCODING_MARKERS
    )


def one_char_token_ratio(text: str) -> float:
    tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        text or "",
    )

    if not tokens:
        return 1.0

    return (
        sum(len(token) == 1 for token in tokens)
        / len(tokens)
    )


## 4. When is extracted text considered suspicious?

Fallback decisions use only transparent technical indicators.

An extraction is flagged when any of the following applies:

- fewer than `MIN_TOTAL_CHARS` characters are extracted;
- for PDFs, fewer than `MIN_PDF_CHARS_PER_PAGE` characters are extracted per page;
- at least `MAX_BAD_ENCODING_MARKERS` obvious encoding-corruption markers occur;
- more than `MAX_ONE_CHAR_TOKEN_RATIO` of alphabetic tokens are single-character
  fragments.

For PDFs, a flag causes the notebook to try the next extraction method. For HTML,
XML and plain text, the same checks only determine whether a file is marked
`success` or `review`.

These checks deliberately avoid language dictionaries, legal-word scores, weighted
candidate rankings, or other opaque quality metrics.

In [6]:
def assess_text(
    text: str,
    *,
    file_format: str,
    n_pages: int | None,
) -> dict[str, Any]:
    text = text or ""

    n_chars = len(text)
    bad_marker_count = count_bad_encoding_markers(text)
    fragment_ratio = one_char_token_ratio(text)

    chars_per_page = None

    if file_format == "pdf" and n_pages:
        chars_per_page = n_chars / n_pages

    reasons: list[str] = []

    if n_chars < MIN_TOTAL_CHARS:
        reasons.append("too_little_text")

    if (
        file_format == "pdf"
        and chars_per_page is not None
        and chars_per_page < MIN_PDF_CHARS_PER_PAGE
    ):
        reasons.append("low_text_per_page")

    if bad_marker_count >= MAX_BAD_ENCODING_MARKERS:
        reasons.append("encoding_corruption")

    if fragment_ratio > MAX_ONE_CHAR_TOKEN_RATIO:
        reasons.append("fragmented_text")

    return {
        "passes": not reasons,
        "reasons": reasons,
        "n_chars": n_chars,
        "n_pages": n_pages,
        "chars_per_page": (
            round(chars_per_page, 1)
            if chars_per_page is not None
            else None
        ),
    }


## 5. HTML, XML and plain-text extraction

HTML-like source files are decoded using explicit encoding declarations when available,
followed by standard fallback detection.

Obvious page chrome such as scripts, navigation, forms, and cookie elements is removed.
The largest plausible document container is retained.

This is intentionally simpler than scoring multiple HTML candidates. Plain-text files
use the same encoding logic and light canonical cleaning.

In [7]:
HTML_REMOVE_SELECTORS = [
    "script",
    "style",
    "noscript",
    "nav",
    "footer",
    "header",
    "form",
    "aside",
    "[class*='cookie']",
    "[id*='cookie']",
    "[class*='share']",
    "[class*='language']",
    "[aria-label*='language']",
    "[class*='pagination']",
    "[class*='print']",
]

HTML_CANDIDATE_SELECTORS = [
    "main",
    "article",
    "#document1",
    "#TexteOnly",
    "#text",
    ".document-content",
    ".document",
    ".content",
    "body",
]


def decode_issue_count(text: str) -> int:
    control_chars = sum(
        1
        for char in text
        if (
            unicodedata.category(char).startswith("C")
            and char not in "\n\t"
        )
    )

    return (
        count_bad_encoding_markers(text)
        + control_chars
    )


def extract_declared_html_encoding(
    data: bytes,
) -> str | None:
    head = data[:8192].decode(
        "ascii",
        errors="ignore",
    )

    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            head,
            flags=re.I,
        )

        if match:
            return match.group(1).strip()

    return None


def decode_bytes_smart(
    data: bytes,
    *,
    is_html: bool = False,
) -> tuple[str, dict[str, Any]]:
    candidates: list[tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(
            (
                "utf-8-sig",
                "bom",
                data.decode(
                    "utf-8-sig",
                    errors="replace",
                ),
            )
        )

    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(
            (
                "utf-16",
                "bom",
                data.decode(
                    "utf-16",
                    errors="replace",
                ),
            )
        )

    declared = (
        extract_declared_html_encoding(data)
        if is_html
        else None
    )

    if declared:
        try:
            candidates.append(
                (
                    declared,
                    "declared",
                    data.decode(
                        declared,
                        errors="replace",
                    ),
                )
            )
        except (LookupError, UnicodeError):
            pass

    try:
        candidates.append(
            (
                "utf-8",
                "strict_utf8",
                data.decode(
                    "utf-8",
                    errors="strict",
                ),
            )
        )
    except UnicodeDecodeError:
        pass

    if is_html and UnicodeDammit is not None:
        try:
            dammit = UnicodeDammit(
                data,
                is_html=True,
                smart_quotes_to=None,
            )

            if dammit.unicode_markup:
                candidates.append(
                    (
                        dammit.original_encoding or "unknown",
                        "unicode_dammit",
                        dammit.unicode_markup,
                    )
                )
        except Exception:
            pass

    if charset_from_bytes is not None:
        try:
            for match in list(
                charset_from_bytes(data)
            )[:3]:
                candidates.append(
                    (
                        getattr(
                            match,
                            "encoding",
                            None,
                        ) or "unknown",
                        "charset_normalizer",
                        str(match),
                    )
                )
        except Exception:
            pass

    for encoding in ("cp1252", "latin-1"):
        try:
            candidates.append(
                (
                    encoding,
                    "fallback",
                    data.decode(
                        encoding,
                        errors="replace",
                    ),
                )
            )
        except Exception:
            pass

    if not candidates:
        candidates.append(
            (
                "utf-8",
                "last_resort",
                data.decode(
                    "utf-8",
                    errors="replace",
                ),
            )
        )

    unique: list[tuple[str, str, str]] = []
    seen: set[str] = set()

    for encoding, source, text in candidates:
        digest = hashlib.sha1(
            text.encode(
                "utf-8",
                errors="ignore",
            )
        ).hexdigest()

        if digest not in seen:
            seen.add(digest)
            unique.append(
                (
                    encoding,
                    source,
                    text,
                )
            )

    encoding, source, text = min(
        unique,
        key=lambda item: decode_issue_count(
            item[2]
        ),
    )

    return normalize_unicode(text), {
        "encoding": encoding,
        "encoding_source": source,
    }


def extract_text_from_html(
    path: Path,
) -> tuple[str, dict[str, Any]]:
    if BeautifulSoup is None:
        raise RuntimeError(
            "BeautifulSoup is not installed."
        )

    html, decode_meta = decode_bytes_smart(
        path.read_bytes(),
        is_html=True,
    )

    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    for selector in HTML_REMOVE_SELECTORS:
        try:
            for tag in soup.select(selector):
                tag.decompose()
        except Exception:
            pass

    candidates: list[tuple[int, str, str]] = []

    for selector in HTML_CANDIDATE_SELECTORS:
        try:
            for tag in soup.select(selector):
                text = tag.get_text(
                    "\n",
                    strip=True,
                )

                if len(text) >= 100:
                    candidates.append(
                        (
                            len(text),
                            selector,
                            text,
                        )
                    )
        except Exception:
            pass

    if candidates:
        _, selected_selector, text = max(
            candidates,
            key=lambda item: item[0],
        )
    else:
        selected_selector = "document"
        text = soup.get_text(
            "\n",
            strip=True,
        )

    return clean_text(text), {
        "method": "html",
        "n_pages": None,
        "html_selected_container": selected_selector,
        **decode_meta,
    }


def extract_text_from_plain(
    path: Path,
) -> tuple[str, dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(
        path.read_bytes(),
        is_html=False,
    )

    return clean_text(text), {
        "method": "plain_text",
        "n_pages": None,
        **decode_meta,
    }


## 6. PDF extraction and OCR fallback

PDFs use a deterministic fallback ladder:

1. **PyMuPDF sorted text** is attempted first;
2. if the result is technically suspicious, **Poppler `pdftotext -layout`** is tried;
3. if both native methods remain suspicious, **MinerU OCR** is attempted.

The methods are not scored against one another. The first extraction satisfying the fixed
technical checks is retained.

MinerU may produce Markdown internally in its temporary output directory. If that happens,
the notebook reads the richest text output, removes common Markdown presentation syntax,
and saves only the final canonical `.txt`. The temporary MinerU files are deleted.

In [8]:
def extract_pdf_pymupdf(
    path: Path,
) -> tuple[str, int | None]:
    if fitz is None:
        raise RuntimeError(
            "PyMuPDF is not installed."
        )

    page_text: list[str] = []

    with fitz.open(path) as document:
        for page in document:
            page_text.append(
                page.get_text(
                    "text",
                    sort=True,
                ) or ""
            )

        n_pages = len(document)

    return (
        clean_text(
            "\n\n".join(page_text)
        ),
        n_pages,
    )


def extract_pdf_pdftotext(
    path: Path,
) -> tuple[str, int | None]:
    if PDFTOTEXT_CLI is None:
        raise RuntimeError(
            "pdftotext is not available on PATH."
        )

    with tempfile.TemporaryDirectory(
        prefix="pdftotext_"
    ) as tmpdir:
        output_path = (
            Path(tmpdir)
            / "output.txt"
        )

        command = [
            PDFTOTEXT_CLI,
            "-layout",
            "-enc",
            "UTF-8",
            str(path),
            str(output_path),
        ]

        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=PDFTOTEXT_TIMEOUT_SECONDS,
            check=False,
        )

        if completed.returncode != 0:
            raise RuntimeError(
                completed.stderr[-2000:]
                or "pdftotext failed"
            )

        text = output_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

    n_pages = None

    if fitz is not None:
        try:
            with fitz.open(path) as document:
                n_pages = len(document)
        except Exception:
            pass

    return clean_text(text), n_pages


def run_mineru(
    input_pdf: Path,
    output_dir: Path,
) -> None:
    if MINERU_CLI is None:
        raise RuntimeError(
            "MinerU is not available on PATH."
        )

    command = [
        MINERU_CLI,
        "-p",
        str(input_pdf),
        "-o",
        str(output_dir),
    ]

    environment = os.environ.copy()
    environment.setdefault(
        "ORT_LOG_SEVERITY_LEVEL",
        "3",
    )

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=OCR_TIMEOUT_SECONDS,
        env=environment,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            completed.stderr[-2000:]
            or "MinerU failed"
        )


def find_mineru_text(
    output_dir: Path,
) -> tuple[Path | None, bool]:
    """
    Return the largest non-empty MinerU text representation.

    The boolean indicates whether the selected intermediate file is Markdown.
    """
    markdown_files = [
        path
        for path in output_dir.glob("**/*.md")
        if (
            path.is_file()
            and path.stat().st_size > 0
        )
    ]

    if markdown_files:
        return (
            max(
                markdown_files,
                key=lambda path: path.stat().st_size,
            ),
            True,
        )

    text_files = [
        path
        for path in output_dir.glob("**/*.txt")
        if (
            path.is_file()
            and path.stat().st_size > 0
        )
    ]

    if text_files:
        return (
            max(
                text_files,
                key=lambda path: path.stat().st_size,
            ),
            False,
        )

    return None, False


def extract_pdf_mineru(
    path: Path,
) -> tuple[str, int | None]:
    with tempfile.TemporaryDirectory(
        prefix="mineru_"
    ) as tmpdir:
        output_dir = Path(tmpdir)

        run_mineru(
            path,
            output_dir,
        )

        extracted_path, is_markdown = find_mineru_text(
            output_dir
        )

        if extracted_path is None:
            raise RuntimeError(
                "MinerU completed but produced no non-empty "
                "text representation."
            )

        text = extracted_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        if is_markdown:
            text = markdown_to_plain_text(text)

    n_pages = None

    if fitz is not None:
        try:
            with fitz.open(path) as document:
                n_pages = len(document)
        except Exception:
            pass

    return clean_text(text), n_pages


def try_pdf_extractor(
    path: Path,
    method: str,
    extractor,
) -> tuple[dict[str, Any] | None, str]:
    try:
        text, n_pages = extractor(path)

        assessment = assess_text(
            text,
            file_format="pdf",
            n_pages=n_pages,
        )

        return {
            "method": method,
            "text": text,
            "assessment": assessment,
        }, ""

    except Exception as exc:
        return None, repr(exc)


## 7. Deterministic fallback and review status

The fallback sequence is evaluated in order.

- If PyMuPDF passes the checks, it is retained.
- Otherwise `pdftotext` is attempted.
- If `pdftotext` also remains suspicious, MinerU OCR is attempted.
- If MinerU produces non-empty text but a simple threshold still triggers, the text is
  retained with `processing_status = review`.
- If OCR is unavailable or fails, the latest non-empty native extraction is retained
  with `processing_status = review`.
- `failed` is reserved for files from which no usable text can be obtained.

This keeps imperfect historical documents visible rather than silently discarding them.

In [9]:
def process_pdf(
    path: Path,
) -> dict[str, Any]:
    attempts: list[dict[str, Any]] = []
    errors: list[str] = []
    fallback_reason: list[str] = []

    # 1. Default native extraction: PyMuPDF
    pymupdf_result, error = try_pdf_extractor(
        path,
        "pymupdf",
        extract_pdf_pymupdf,
    )

    if error:
        errors.append(
            f"pymupdf: {error}"
        )
        fallback_reason.append(
            "pymupdf_failed"
        )

    elif pymupdf_result is not None:
        attempts.append(
            pymupdf_result
        )

        if pymupdf_result[
            "assessment"
        ]["passes"]:
            return {
                "selected": pymupdf_result,
                "status": "success",
                "review_reason": "",
                "ocr_used": False,
                "fallback_reason": "",
                "errors": errors,
            }

        fallback_reason.extend(
            f"pymupdf:{reason}"
            for reason in pymupdf_result[
                "assessment"
            ]["reasons"]
        )

    # 2. Native fallback: pdftotext -layout
    if (
        RUN_PDFTOTEXT
        and PDFTOTEXT_CLI is not None
    ):
        poppler_result, error = try_pdf_extractor(
            path,
            "pdftotext",
            extract_pdf_pdftotext,
        )

        if error:
            errors.append(
                f"pdftotext: {error}"
            )
            fallback_reason.append(
                "pdftotext_failed"
            )

        elif poppler_result is not None:
            attempts.append(
                poppler_result
            )

            if poppler_result[
                "assessment"
            ]["passes"]:
                return {
                    "selected": poppler_result,
                    "status": "success",
                    "review_reason": "",
                    "ocr_used": False,
                    "fallback_reason": "; ".join(
                        fallback_reason
                    ),
                    "errors": errors,
                }

            fallback_reason.extend(
                f"pdftotext:{reason}"
                for reason in poppler_result[
                    "assessment"
                ]["reasons"]
            )

    elif RUN_PDFTOTEXT:
        fallback_reason.append(
            "pdftotext_unavailable"
        )

    # 3. OCR fallback: MinerU
    if (
        RUN_OCR
        and MINERU_CLI is not None
    ):
        ocr_result, error = try_pdf_extractor(
            path,
            "mineru_ocr",
            extract_pdf_mineru,
        )

        if error:
            errors.append(
                f"mineru_ocr: {error}"
            )
            fallback_reason.append(
                "mineru_failed"
            )

        elif (
            ocr_result is not None
            and ocr_result["text"].strip()
        ):
            attempts.append(
                ocr_result
            )

            if ocr_result[
                "assessment"
            ]["passes"]:
                return {
                    "selected": ocr_result,
                    "status": "success",
                    "review_reason": "",
                    "ocr_used": True,
                    "fallback_reason": "; ".join(
                        fallback_reason
                    ),
                    "errors": errors,
                }

            return {
                "selected": ocr_result,
                "status": "review",
                "review_reason": "; ".join(
                    f"mineru:{reason}"
                    for reason in ocr_result[
                        "assessment"
                    ]["reasons"]
                ),
                "ocr_used": True,
                "fallback_reason": "; ".join(
                    fallback_reason
                ),
                "errors": errors,
            }

    elif RUN_OCR:
        fallback_reason.append(
            "mineru_unavailable"
        )

    # If nothing passed, keep the latest non-empty native extraction.
    nonempty_attempts = [
        attempt
        for attempt in attempts
        if attempt.get(
            "text",
            "",
        ).strip()
    ]

    if nonempty_attempts:
        selected = nonempty_attempts[-1]

        return {
            "selected": selected,
            "status": "review",
            "review_reason": "; ".join(
                f"{selected['method']}:{reason}"
                for reason in selected[
                    "assessment"
                ]["reasons"]
            ) or "no_method_passed_checks",
            "ocr_used": (
                selected["method"]
                == "mineru_ocr"
            ),
            "fallback_reason": "; ".join(
                fallback_reason
            ),
            "errors": errors,
        }

    return {
        "selected": None,
        "status": "failed",
        "review_reason": "no_text_extracted",
        "ocr_used": False,
        "fallback_reason": "; ".join(
            fallback_reason
        ),
        "errors": errors,
    }


def process_non_pdf(
    path: Path,
    file_format: str,
) -> dict[str, Any]:
    extractor = (
        extract_text_from_html
        if file_format in {
            "html",
            "htm",
            "xhtml",
            "xml",
        }
        else extract_text_from_plain
    )

    try:
        text, meta = extractor(path)

        assessment = assess_text(
            text,
            file_format=file_format,
            n_pages=None,
        )

    except Exception as exc:
        return {
            "selected": None,
            "status": "failed",
            "review_reason": "extraction_failed",
            "ocr_used": False,
            "fallback_reason": "",
            "errors": [repr(exc)],
        }

    selected = {
        "method": meta["method"],
        "text": text,
        "assessment": assessment,
    }

    if not text.strip():
        status = "failed"
        review_reason = "no_text_extracted"

    elif assessment["passes"]:
        status = "success"
        review_reason = ""

    else:
        status = "review"
        review_reason = "; ".join(
            assessment["reasons"]
        )

    return {
        "selected": (
            selected
            if text.strip()
            else None
        ),
        "status": status,
        "review_reason": review_reason,
        "ocr_used": False,
        "fallback_reason": "",
        "errors": [],
    }


## 8. Resumable processing state

`logs/com_manual/processing_state.jsonl` is an append-only operational checkpoint.

A previous `success` or `review` result is reused only when:

- the processing version is unchanged;
- the current raw SHA-256 matches the previous run;
- the expected `.txt` output exists;
- the processed-text SHA-256 still matches the checkpoint.

Failed records are retried automatically. Set `FORCE_REPROCESS = True` to rebuild
everything deliberately.

This makes long OCR runs restartable without adding extra audit tables to the publication
outputs.

In [10]:
def json_safe(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(
        value,
        (str, int, float, bool),
    ):
        if (
            isinstance(value, float)
            and pd.isna(value)
        ):
            return None

        return value

    if isinstance(value, Path):
        return value.as_posix()

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return str(value)


def load_processing_state() -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}

    if not PROCESSING_STATE_JSONL.exists():
        return latest

    with PROCESSING_STATE_JSONL.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line in handle:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # Ignore a malformed partial final line after interruption.
                continue

            record_id = str(
                record.get(
                    "record_id",
                    "",
                )
            ).strip()

            if record_id:
                latest[record_id] = record

    return latest


def append_processing_state(
    record: dict[str, Any],
) -> None:
    with PROCESSING_STATE_JSONL.open(
        "a",
        encoding="utf-8",
    ) as handle:
        handle.write(
            json.dumps(
                json_safe(record),
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )


def expected_text_path(
    raw_path: Path,
) -> Path:
    return (
        PROCESSED_TEXT_DIR
        / raw_path.with_suffix(".txt").name
    )


def state_is_reusable(
    state_record: dict[str, Any] | None,
    *,
    raw_sha256: str,
    text_path: Path,
) -> bool:
    if FORCE_REPROCESS or not state_record:
        return False

    if state_record.get(
        "processing_status"
    ) not in {"success", "review"}:
        return False

    if (
        state_record.get("processing_version")
        != PROCESSING_VERSION
    ):
        return False

    if (
        str(
            state_record.get(
                "raw_sha256",
                "",
            )
        ).lower()
        != raw_sha256.lower()
    ):
        return False

    if (
        state_record.get("clean_file_path")
        != repo_path(text_path)
    ):
        return False

    if (
        not text_path.exists()
        or text_path.stat().st_size <= 0
    ):
        return False

    expected_text_sha = str(
        state_record.get(
            "clean_sha256",
            "",
        )
    ).lower()

    if not re.fullmatch(
        r"[0-9a-f]{64}",
        expected_text_sha,
    ):
        return False

    return (
        sha256_file(text_path)
        == expected_text_sha
    )


def process_file(
    input_row: dict[str, Any],
    *,
    raw_sha256: str,
) -> dict[str, Any]:
    record_id = str(
        input_row["record_id"]
    )

    raw_path = Path(
        input_row["raw_path_internal"]
    )

    file_format = str(
        input_row["file_format_normalized"]
    ).strip().lower()

    text_path = expected_text_path(
        raw_path
    )

    base = {
        "timestamp_utc": utc_now_iso(),
        "processing_version": PROCESSING_VERSION,
        "record_id": record_id,
        "raw_file_name": raw_path.name,
        "raw_file_path": repo_path(raw_path),
        "raw_sha256": raw_sha256,
    }

    if file_format not in SUPPORTED_FILE_FORMATS:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": "",
            "review_reason": "unsupported_file_format",
            "clean_file_name": "",
            "clean_file_path": "",
            "clean_sha256": "",
            "n_pages": None,
            "n_chars": 0,
            "chars_per_page": None,
            "error": (
                "unsupported_file_format:"
                f"{file_format}"
            ),
        }

    result = (
        process_pdf(raw_path)
        if file_format == "pdf"
        else process_non_pdf(
            raw_path,
            file_format,
        )
    )

    selected = result["selected"]

    if selected is None:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": result.get(
                "fallback_reason",
                "",
            ),
            "review_reason": result.get(
                "review_reason",
                "no_text_extracted",
            ),
            "clean_file_name": "",
            "clean_file_path": "",
            "clean_sha256": "",
            "n_pages": None,
            "n_chars": 0,
            "chars_per_page": None,
            "error": (
                " | ".join(
                    result.get(
                        "errors",
                        [],
                    )
                )
                or "no_text_extracted"
            ),
        }

    text = selected["text"]

    text_path.write_text(
        text,
        encoding="utf-8",
    )

    assessment = selected[
        "assessment"
    ]

    return {
        **base,
        "processing_status": result["status"],
        "extraction_method": selected["method"],
        "ocr_used": bool(
            result["ocr_used"]
        ),
        "fallback_reason": result.get(
            "fallback_reason",
            "",
        ),
        "review_reason": result.get(
            "review_reason",
            "",
        ),
        "clean_file_name": text_path.name,
        "clean_file_path": repo_path(text_path),
        "clean_sha256": sha256_file(text_path),
        "n_pages": assessment.get("n_pages"),
        "n_chars": assessment.get("n_chars", len(text)),
        "chars_per_page": assessment.get("chars_per_page"),
        "error": " | ".join(
            result.get(
                "errors",
                [],
            )
        ),
    }


## 9. Process the COM manual collection

Each successfully downloaded source file is handled once.

Before extraction or checkpoint reuse, its current SHA-256 is compared with the SHA-256
recorded by notebook 08. A mismatch stops the run rather than silently processing altered
raw data.

The progress bar reports successful files, review files, failures, OCR selections, and
checkpoint rows reused during the current run.

In [11]:
processing_state = load_processing_state()
manifest_rows: list[dict[str, Any]] = []

run_counts = {
    "success": 0,
    "review": 0,
    "failed": 0,
    "ocr": 0,
    "reused": 0,
}

target_rows = work_manifest.to_dict(
    "records"
)

progress = tqdm(
    target_rows,
    desc="Processing COM manual files",
    unit="file",
    dynamic_ncols=True,
)

for input_row in progress:
    record_id = str(
        input_row["record_id"]
    )

    raw_path = Path(
        input_row["raw_path_internal"]
    )

    actual_raw_sha256 = sha256_file(
        raw_path
    )

    expected_raw_sha256 = str(
        input_row["raw_sha256"]
    ).lower()

    if actual_raw_sha256 != expected_raw_sha256:
        raise ValueError(
            "Raw SHA-256 mismatch for "
            f"{record_id}: {repo_path(raw_path)}"
        )

    text_path = expected_text_path(
        raw_path
    )

    previous_state = processing_state.get(
        record_id
    )

    if state_is_reusable(
        previous_state,
        raw_sha256=actual_raw_sha256,
        text_path=text_path,
    ):
        state_record = previous_state
        run_counts["reused"] += 1

    else:
        state_record = process_file(
            input_row,
            raw_sha256=actual_raw_sha256,
        )

        append_processing_state(
            state_record
        )

        processing_state[
            record_id
        ] = state_record

    status = state_record.get(
        "processing_status",
        "failed",
    )

    if status in {
        "success",
        "review",
        "failed",
    }:
        run_counts[status] += 1
    else:
        run_counts["failed"] += 1

    if state_record.get("ocr_used"):
        run_counts["ocr"] += 1

    manifest_rows.append({
        "record_id": record_id,
        "source_row": input_row["source_row"],
        "url_index_in_row": input_row["url_index_in_row"],
        "decision_date": input_row["decision_date"],
        "case_numbers_raw": input_row["case_numbers_raw"],
        "case_numbers_normalized": input_row["case_numbers_normalized"],
        "decision_label": input_row["decision_label"],
        "decision_title": input_row["decision_title"],
        "celex": input_row["celex"],
        "source_url": input_row["source_url"],
        "resolved_url": input_row.get("resolved_url", ""),
        "raw_file_name": raw_path.name,
        "raw_file_path": repo_path(raw_path),
        "clean_file_name": state_record.get(
            "clean_file_name",
            "",
        ),
        "clean_file_path": state_record.get(
            "clean_file_path",
            "",
        ),
        "extraction_method": state_record.get(
            "extraction_method",
            "",
        ),
        "ocr_used": state_record.get(
            "ocr_used",
            False,
        ),
        "processing_status": status,
        "review_reason": state_record.get(
            "review_reason",
            "",
        ),
        "fallback_reason": state_record.get(
            "fallback_reason",
            "",
        ),
        "raw_sha256": actual_raw_sha256,
        "clean_sha256": state_record.get(
            "clean_sha256",
            "",
        ),
        "n_pages": state_record.get(
            "n_pages",
        ),
        "n_chars": state_record.get(
            "n_chars",
            0,
        ),
        "processing_version": state_record.get(
            "processing_version",
            PROCESSING_VERSION,
        ),
        "error": state_record.get(
            "error",
            "",
        ),
    })

    progress.set_postfix(
        {
            "ok": run_counts["success"],
            "review": run_counts["review"],
            "fail": run_counts["failed"],
            "ocr": run_counts["ocr"],
            "reused": run_counts["reused"],
        },
        refresh=False,
    )


processing_manifest = (
    pd.DataFrame(manifest_rows)
    .sort_values(
        ["source_row", "url_index_in_row"]
    )
    .reset_index(drop=True)
)

print(f"Files considered: {len(processing_manifest):,}")
print(f"Successful: {run_counts['success']:,}")
print(f"Review: {run_counts['review']:,}")
print(f"Failed: {run_counts['failed']:,}")
print(f"OCR selected: {run_counts['ocr']:,}")
print(f"Checkpoints reused this run: {run_counts['reused']:,}")


Processing COM manual files:   0%|          | 0/16 [00:00<?, ?file/s]

Files considered: 16
Successful: 16
Review: 0
Failed: 0
OCR selected: 0
Checkpoints reused this run: 0


## 10. Save the clean-file manifest

The publication-oriented manifest is intentionally compact. It connects each source record
to its raw file and canonical text file, while retaining only the processing information
that is useful for provenance or human review.

There is no candidate manifest, separate quality table, Markdown copy, or candidate-text
directory.

The most important downstream fields are:

- `clean_file_name`
- `clean_file_path`
- `processing_status`

Rows with `processing_status == "success"` are ordinary processed files. Rows marked
`review` contain usable text but triggered one of the simple technical checks. Rows marked
`failed` did not yield usable text.

In [12]:
processing_manifest.to_csv(
    CLEAN_FILE_MANIFEST,
    index=False,
    encoding="utf-8",
)

print(
    "Saved:",
    repo_path(CLEAN_FILE_MANIFEST),
)

display(
    processing_manifest[
        [
            "record_id",
            "decision_title",
            "celex",
            "clean_file_name",
            "clean_file_path",
            "extraction_method",
            "ocr_used",
            "processing_status",
        ]
    ].head(20)
)


summary = (
    processing_manifest
    .groupby(
        [
            "processing_status",
            "extraction_method",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="files")
    .sort_values(
        [
            "processing_status",
            "extraction_method",
        ]
    )
)

print("\nProcessing summary:")
display(summary)


review_rows = processing_manifest.loc[
    processing_manifest[
        "processing_status"
    ].isin(
        ["review", "failed"]
    )
].copy()

if len(review_rows):
    print(
        f"\nRows requiring attention: {len(review_rows):,}"
    )

    display(
        review_rows[
            [
                "record_id",
                "decision_title",
                "celex",
                "raw_file_name",
                "extraction_method",
                "processing_status",
                "review_reason",
                "fallback_reason",
                "error",
            ]
        ].head(30)
    )

else:
    print("\nNo rows require review.")


Saved: output/com_manual/com_manual_clean_file_manifest.csv


,record_id,decision_title,celex,clean_file_name,clean_file_path,extraction_method,ocr_used,processing_status
0,com_manual__row00001_url01,National Panasonic France,31982D0465,31982D0465.txt,data/processed/com_manual/text/31982D0465.txt,html,False,success
1,com_manual__row00010_url01,GSM Spain,31997D0181,31997D0181.txt,data/processed/com_manual/text/31997D0181.txt,html,False,success
2,com_manual__row00011_url01,Additional implementation periods to Portugal,31997D0310,31997D0310.txt,data/processed/com_manual/text/31997D0310.txt,html,False,success
3,com_manual__row00012_url01,Additional implementation periods to Luxembourg,31997D0568,31997D0568.txt,data/processed/com_manual/text/31997D0568.txt,html,False,success
4,com_manual__row00013_url01,Additional implementation periods to Spain,31997D0603,31997D0603.txt,data/processed/com_manual/text/31997D0603.txt,html,False,success
5,com_manual__row00014_url01,Broadcasting television Flanders,31997D0606,31997D0606.txt,data/processed/com_manual/text/31997D0606.txt,pymupdf,False,success
6,com_manual__row00015_url01,Additional implementation periods to Greece,31997D0607,31997D0607.txt,data/processed/com_manual/text/31997D0607.txt,html,False,success
7,com_manual__row00016_url01,Italian ports legislation | Tariffs for piloti...,31997D0744,31997D0744.txt,data/processed/com_manual/text/31997D0744.txt,html,False,success
8,com_manual__row00002_url01,Anheuser-Busch Inc/Scottish and New Castle,32000D0146,32000D0146.txt,data/processed/com_manual/text/32000D0146.txt,html,False,success
9,com_manual__row00003_url01,Zaventem,31995D0364,31995D0364.txt,data/processed/com_manual/text/31995D0364.txt,html,False,success



Processing summary:


,processing_status,extraction_method,files
0,success,html,15
1,success,pymupdf,1



No rows require review.


## 11. Scientific-data publication notes

This notebook intentionally keeps the processing description easier to reproduce than the
older candidate-ranking approach.

For a methods or data-descriptor paper, the processing step can be described directly:

> Successfully downloaded Commission source files were converted to canonical UTF-8 plain
> text. HTML and text documents were decoded and lightly normalized. PDFs were processed
> using a deterministic fallback hierarchy consisting of PyMuPDF native extraction,
> Poppler `pdftotext -layout`, and MinerU OCR when native extraction failed simple technical
> checks. One processed text file was retained per raw source file.

The raw documents are never modified. `raw_sha256` and `clean_sha256` identify the exact
input and processed files used in this stage.

For release validation, a practical human check is to inspect:

- every file where `ocr_used == True`;
- every row with `processing_status == "review"`;
- a small sample of ordinary native-text extractions across the time span of the collection.

Package and external-tool versions should be frozen once at repository/release level
(for example in `requirements.txt`, `environment.yml`, or the repository documentation)
rather than repeated in every manifest row.